In [ ]:
# ============================================================
# 1. Hardware Verification & GPU Diagnostics
# ============================================================
import torch
import sys
from IPython.display import HTML, display

if not torch.cuda.is_available():
    display(HTML("<div style='background-color:#ffebee;padding:12px;border-radius:8px;border-left:5px solid #f44336;'><h3 style='color:#c62828;margin:0;'>⚠️ GPU Not Detected!</h3><p style='margin:5px 0 0 0;'>Please enable <b>GPU</b> in <b>Settings > Accelerator</b>.</p></div>"))
    raise SystemExit("GPU accelerator missing.")

gpu_name = torch.cuda.get_device_name(0)
vram_gb = round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2)
cuda_ver = torch.version.cuda

display(HTML(f"""
<div style='background-color:#e8f5e9;padding:15px;border-radius:8px;border-left:5px solid #4caf50;'>
  <h3 style='color:#2e7d32;margin:0 0 8px 0;'>✅ Hardware Accelerator Ready</h3>
  <table style='font-family:monospace;font-size:13px;'>
    <tr><td><b>GPU Device:</b></td><td>&nbsp;{gpu_name}</td></tr>
    <tr><td><b>Total VRAM:</b></td><td>&nbsp;{vram_gb} GB</td></tr>
    <tr><td><b>CUDA Version:</b></td><td>&nbsp;{cuda_ver}</td></tr>
    <tr><td><b>PyTorch:</b></td><td>&nbsp;{torch.__version__}</td></tr>
  </table>
</div>
"""))

In [ ]:
# ============================================================
# 2. Install ComfyUI Core & Audio Dependencies
# ============================================================
import os
from IPython.display import HTML, display

print("🚀 Setting up ComfyUI audio environment...")

# Change working directory to /kaggle/working
os.chdir('/kaggle/working')

if not os.path.exists("/kaggle/working/ComfyUI"):
    !git clone https://github.com/comfyanonymous/ComfyUI.git /kaggle/working/ComfyUI
    
%cd /kaggle/working/ComfyUI

!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q -r requirements.txt
!pip install -q soundfile torchaudio accelerate sentencepiece mutagen websocket-client safetensors
!apt-get -y install -qq aria2 ffmpeg > /dev/null 2>&1

display(HTML("<div style='background-color:#e3f2fd;padding:12px;border-radius:8px;border-left:5px solid #2196f3;'><b style='color:#1565c0;'>✅ Environment installed and ready!</b></div>"))

In [ ]:
# ============================================================
# 3. Download FP16 Model & Launch Fast Daemon
# ============================================================
import os
import time
import subprocess
import urllib.request
from IPython.display import HTML, display

%cd /kaggle/working/ComfyUI

model_dir = "/kaggle/working/ComfyUI/models/checkpoints"
os.makedirs(model_dir, exist_ok=True)
model_path = os.path.join(model_dir, "yue2_3b_fp16.safetensors")

# Download pre-converted FP16 Checkpoint directly (no conversion needed)
if not os.path.exists(model_path) or os.path.getsize(model_path) < 6_000_000_000:
    print("📥 Downloading YuE2 FP16 checkpoint (~7.26 GB) from Jokality/YuE2-fp16... Please wait.")
    !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M \
      "https://huggingface.co/Jokality/YuE2-fp16/resolve/main/checkpoints/yue2_3b_fp16.safetensors" \
      -d "{model_dir}" \
      -o "yue2_3b_fp16.safetensors"
else:
    print("⚡ Found existing FP16 checkpoint in cache.")

# Terminate previous instances to avoid port conflicts
!pkill -9 -f "main.py" || True
time.sleep(2)

print("🔄 Starting ComfyUI Turbo Daemon with native FP16 acceleration...")
server_log = open("/kaggle/working/comfy_server.log", "w")

server_cmd = [\
    "python", "main.py",\
    "--listen", "127.0.0.1",\
    "--port", "8188",\
    "--force-fp16",\
    "--highvram",\
    "--dont-upcast-attention"\
]

proc = subprocess.Popen(server_cmd, stdout=server_log, stderr=server_log, cwd="/kaggle/working/ComfyUI")

server_alive = False
for i in range(40):
    if proc.poll() is not None:
        break
    try:
        urllib.request.urlopen("http://127.0.0.1:8188/system_stats", timeout=2)
        server_alive = True
        break
    except Exception:
        time.sleep(1.5)

if server_alive:
    display(HTML("""
    <div style='background-color:#e8f5e9;padding:15px;border-radius:8px;border-left:5px solid #4caf50;'>
      <b style='color:#2e7d32;'>🟢 YuE2 Native FP16 Engine Online!</b><br>
      <span style='color:#555;'>Model: <code>yue2_3b_fp16.safetensors</code> • Hardware Tensor Cores Active. Run <b>Block 4</b>.</span>
    </div>
    """))
else:
    print("\n❌ Startup failed. Log contents (/kaggle/working/comfy_server.log):\n" + "-"*60)
    with open("/kaggle/working/comfy_server.log", "r") as f:
        print(f.read())
    print("-" * 60)

In [ ]:
# ============================================================
# 4. Interactive Gradio Interface for YuE2 Music Generation
# ============================================================
!pip install -q gradio > /dev/null 2>&1

import os
import json
import time
import random
import glob
import uuid
import subprocess
import websocket
import urllib.request
import gradio as gr
from IPython.display import HTML, display
import base64

print("🎨 Setting up Gradio interface...")

# Preset styles
preset_dict = {
    "Indie Pop Anthem": "Upbeat indie pop with warm female vocals, bright electric guitars, punchy drums, melodic bass, and subtle synth layers. Catchy and energetic, with an uplifting summer atmosphere, a memorable chorus, and polished modern production.",
    "Synthwave 80s Retro": "1980s retro synthwave, analog vintage synthesizers, gated snare reverb, driving arpeggiated bassline, male nostalgic vocals, neon aesthetic, cinematic nocturnal drive.",
    "Lo-Fi Chill Hop": "Warm lofi hip hop, vinyl crackle, mellow electric piano chords, smooth relaxed boom bap drums, deep warm sub bass, peaceful afternoon coffee shop atmosphere.",
    "Cinematic Epic Orchestral": "Epic cinematic trailer music, soaring brass ensemble, thundering Taiko drums, expressive emotional strings, dramatic crescendos, heroic orchestral climax.",
    "Modern Alt Rock": "Modern alternative rock, overdriven crunchy guitars, aggressive live acoustic drum kit, energetic lead vocals, heavy bassline, anthemic chorus."
}

def generate_music(genre_preset, custom_style, lyrics, enable_abc, composition_mode, 
                   max_abc_tokens, sampler_name, scheduler, steps, randomize_seed, 
                   seed, audio_format, audio_quality, progress=gr.Progress()):
    """Main generation function"""
    
    if not lyrics or not lyrics.strip():
        return None, "❌ Please enter lyrics!"
    
    progress(0, desc="Initializing...")
    
    active_style = custom_style if genre_preset == "Custom" else preset_dict.get(genre_preset, custom_style)
    active_seed = random.randint(1, 2147483647) if randomize_seed else seed
    ckpt_name = "yue2_3b_fp16.safetensors"
    MAX_DURATION_SECONDS = 360
    
    workflow_api = {
      "10": {
        "inputs": {
          "filename_prefix": f"audio/YuE2_Gradio_{int(time.time())}",
          "format": audio_format,
          "format.quality": audio_quality,
          "audio": ["33:9", 0]
        },
        "class_type": "SaveAudioAdvanced"
      },
      "33:15": {
        "inputs": {"ckpt_name": ckpt_name},
        "class_type": "CheckpointLoaderSimple"
      },
      "33:24": {
        "inputs": {
          "style": ["33:36", 0],
          "lyrics": ["33:37", 0],
          "seed": ["33:34", 0],
          "mode": composition_mode,
          "max_abc_tokens": max_abc_tokens,
          "temperature": 0.7,
          "top_p": 0.9,
          "top_k": 30,
          "repetition_penalty": 1.005,
          "penalty_window": 100,
          "clip": ["33:15", 1]
        },
        "class_type": "YuE2GenerateABC"
      },
      "33:14": {
        "inputs": {"source": ["33:30", 0]},
        "class_type": "PreviewAny"
      },
      "33:30": {
        "inputs": {
          "switch": ["33:31", 0],
          "on_false": ["33:32", 0],
          "on_true": ["33:24", 0]
        },
        "class_type": "ComfySwitchNode"
      },
      "33:32": {
        "inputs": {"value": ""},
        "class_type": "PrimitiveString"
      },
      "33:25": {
        "inputs": {
          "style": ["33:36", 0],
          "lyrics": ["33:37", 0],
          "abc": ["33:14", 0],
          "seed": ["33:34", 0],
          "mode": composition_mode,
          "max_duration": MAX_DURATION_SECONDS,
          "temperature": 1.0,
          "top_p": 0.95,
          "top_k": 100,
          "repetition_penalty": 1.2,
          "clip": ["33:15", 1]
        },
        "class_type": "YuE2GenerateMusic"
      },
      "33:31": {
        "inputs": {"value": enable_abc},
        "class_type": "PrimitiveBoolean"
      },
      "33:18": {
        "inputs": {"conditioning": ["33:25", 0]},
        "class_type": "ConditioningZeroOut"
      },
      "33:23": {
        "inputs": {"source": ["33:25", 1]},
        "class_type": "PreviewAny"
      },
      "33:8": {
        "inputs": {
          "seed": ["33:34", 0],
          "steps": steps,
          "cfg": 1.0,
          "sampler_name": sampler_name,
          "scheduler": scheduler,
          "denoise": 1.0,
          "model": ["33:15", 0],
          "positive": ["33:25", 0],
          "negative": ["33:18", 0],
          "latent_image": ["33:5", 0]
        },
        "class_type": "KSampler"
      },
      "33:5": {
        "inputs": {
          "seconds": ["33:25", 1],
          "batch_size": 1
        },
        "class_type": "EmptyYuE2LatentAudio"
      },
      "33:9": {
        "inputs": {
          "samples": ["33:8", 0],
          "vae": ["33:15", 2]
        },
        "class_type": "VAEDecodeAudio"
      },
      "33:34": {
        "inputs": {"seed": active_seed},
        "class_type": "SeedNode"
      },
      "33:36": {
        "inputs": {"value": active_style},
        "class_type": "PrimitiveStringMultiline"
      },
      "33:37": {
        "inputs": {"value": lyrics},
        "class_type": "PrimitiveStringMultiline"
      }
    }
    
    node_names = {
        "33:15": f"CheckpointLoaderSimple ({ckpt_name})",
        "33:24": "YuE2GenerateABC (Symbolic Score Planner)",
        "33:30": "ComfySwitchNode (ABC Bypass Route)",
        "33:25": "YuE2GenerateMusic (Acoustic Audio Synthesis)",
        "33:8": f"KSampler (Diffusion Decoding - {steps} steps)",
        "33:9": "VAEDecodeAudio (Audio Decompression)",
        "10": "SaveAudioAdvanced (Export)"
    }
    
    try:
        progress(0.1, desc="Connecting to ComfyUI...")
        client_id = str(uuid.uuid4())
        ws = websocket.WebSocket()
        ws.connect(f"ws://127.0.0.1:8188/ws?clientId={client_id}")
        
        req_payload = json.dumps({"prompt": workflow_api, "client_id": client_id}).encode('utf-8')
        req = urllib.request.Request("http://127.0.0.1:8188/prompt", data=req_payload, headers={'Content-Type': 'application/json'})
        resp = urllib.request.urlopen(req)
        prompt_id = json.loads(resp.read())['prompt_id']
        
        print(f"📡 [Prompt ID: {prompt_id}] Active Seed: {active_seed}")
        
        start_time = time.time()
        job_failed = False
        result_file = None
        
        progress(0.2, desc="Generating music...")
        
        while True:
            msg = ws.recv()
            if isinstance(msg, str):
                event = json.loads(msg)
                event_type = event.get('type')
                event_data = event.get('data', {})
                
                if event_type == 'execution_error':
                    error_msg = event_data.get('exception_message', 'Unknown error')
                    print(f"\n❌ Pipeline Error: {error_msg}")
                    job_failed = True
                    break
                
                if event_type == 'executing':
                    node_id = event_data.get('node')
                    if node_id is None:
                        if event_data.get('prompt_id') == prompt_id:
                            print("\n\n🎉 Pipeline Completed!")
                            progress(1.0, desc="Complete!")
                            break
                    else:
                        stage_label = node_names.get(str(node_id), f"Node {node_id}")
                        progress(0.3, desc=f"Stage: {stage_label}")
                        print(f"\n▶️ [Stage: {stage_label}]")
                
                elif event_type == 'progress':
                    curr_val = event_data.get('value', 0)
                    max_val = event_data.get('max', steps)
                    pct = curr_val / max_val
                    gradio_progress = 0.3 + (pct * 0.65)
                    progress(gradio_progress, desc=f"Sampling: {curr_val}/{max_val} ({int(pct*100)}%)")
        
        ws.close()
        total_seconds = round(time.time() - start_time, 1)
        print(f"⏱️ Total time: {total_seconds}s")
        
        if not job_failed:
            progress(0.95, desc="Loading audio file...")
            candidate_files = glob.glob(f"/kaggle/working/ComfyUI/output/audio/YuE2_Gradio_*.{audio_format}")
            
            if candidate_files:
                latest_file = max(candidate_files, key=os.path.getctime)
                result_file = latest_file
                print(f"✅ Generated: {latest_file}")
                progress(1.0, desc="Done!")
                return result_file, f"✅ Success! Generated in {total_seconds}s | Seed: {active_seed}"
            else:
                return None, "❌ No output file found"
        else:
            return None, f"❌ Generation failed"
    
    except Exception as e:
        print(f"❌ Error: {e}")
        import traceback
        traceback.print_exc()
        return None, f"❌ Error: {str(e)}"

# Create Gradio Interface
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("""
    # 🎵 YuE2 Music Studio
    ### AI-Powered Text-to-Music Generation
    
    Create music from text descriptions and lyrics using YuE2 model.
    """)
    
    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### 🎼 Style & Lyrics")
            
            genre_preset = gr.Dropdown(
                choices=["Custom", "Indie Pop Anthem", "Synthwave 80s Retro", 
                        "Lo-Fi Chill Hop", "Cinematic Epic Orchestral", "Modern Alt Rock"],
                value="Indie Pop Anthem",
                label="Genre Preset"
            )
            
            custom_style = gr.Textbox(
                value=preset_dict["Indie Pop Anthem"],
                label="Custom Style Description",
                lines=3,
                placeholder="Describe your desired music style..."
            )
            
            lyrics = gr.Textbox(
                value="""[Verse]
Morning light across the window
City waking down below
I can hear the streets are calling
Feels like somewhere we should go

[Chorus]
Run with me into the sunlight
Leave the shadows far behind
We don't need to know tomorrow
Tonight the whole world feels alive

[Verse]
Radio playing through the open door
Laughing like we did before
Every mile becomes a memory
And I just want a little more

[Chorus]
Run with me into the sunlight
Leave the shadows far behind
We don't need to know tomorrow
Tonight the whole world feels alive

[Outro]""",
                label="Lyrics (use [Verse], [Chorus], [Bridge], [Outro] tags)",
                lines=15
            )
        
        with gr.Column(scale=1):
            gr.Markdown("### ⚙️ Generation Settings")
            
            composition_mode = gr.Radio(
                choices=["full", "melody"],
                value="full",
                label="Composition Mode"
            )
            
            enable_abc = gr.Checkbox(
                value=False,
                label="Enable ABC Planning (Symbolic Score)",
                info="Uncheck for faster direct audio generation (saves 2-3 min)"
            )
            
            max_abc_tokens = gr.Dropdown(
                choices=[1024, 2048, 4096, 8192],
                value=8192,
                label="Max ABC Tokens"
            )
            
            sampler_name = gr.Dropdown(
                choices=["dpm_2", "euler", "dpmpp_2m", "heun", "dpmpp_sde"],
                value="dpm_2",
                label="Sampler"
            )
            
            scheduler = gr.Dropdown(
                choices=["sgm_uniform", "normal", "karras", "simple"],
                value="sgm_uniform",
                label="Scheduler"
            )
            
            steps = gr.Slider(
                minimum=10,
                maximum=50,
                value=32,
                step=1,
                label="Sampling Steps"
            )
            
            randomize_seed = gr.Checkbox(
                value=True,
                label="Randomize Seed"
            )
            
            seed = gr.Number(
                value=42,
                label="Seed",
                interactive=False
            )
            
            audio_format = gr.Radio(
                choices=["mp3", "wav", "flac"],
                value="mp3",
                label="Audio Format"
            )
            
            audio_quality = gr.Dropdown(
                choices=["320k", "256k", "192k", "128k"],
                value="320k",
                label="Audio Quality"
            )
    
    generate_btn = gr.Button("🎵 Generate Music", variant="primary", size="lg")
    
    with gr.Row():
        audio_output = gr.Audio(label="Generated Music", type="filepath")
        status_output = gr.Textbox(label="Status", interactive=False)
    
    def update_style(preset):
        if preset != "Custom":
            return preset_dict.get(preset, "")
        return gr.update()
    
    def toggle_seed(randomize):
        return gr.update(interactive=not randomize)
    
    genre_preset.change(fn=update_style, inputs=[genre_preset], outputs=[custom_style])
    randomize_seed.change(fn=toggle_seed, inputs=[randomize_seed], outputs=[seed])
    
    generate_btn.click(
        fn=generate_music,
        inputs=[
            genre_preset, custom_style, lyrics, enable_abc, composition_mode,
            max_abc_tokens, sampler_name, scheduler, steps, randomize_seed,
            seed, audio_format, audio_quality
        ],
        outputs=[audio_output, status_output]
    )

# Launch Gradio with PUBLIC link
demo.launch(share=True, debug=False)

In [ ]:
# ============================================================
# 5. File Manager: Download All Generated Tracks
# ============================================================
import glob
import os
import shutil
import time
from IPython.display import FileLink, HTML, display

output_dir = "/kaggle/working/ComfyUI/output/audio"

# Check if directory exists
if not os.path.exists(output_dir):
    display(HTML("<div style='background-color:#fff3cd;padding:15px;border-radius:8px;border-left:5px solid #ffc107;'><b style='color:#856404;'>⚠️ Output directory not found.</b><br><span style='color:#856404;'>Generate some tracks first using the Web Interface (Block 4).</span></div>"))
else:
    all_audio = glob.glob(f"{output_dir}/*.*")
    
    if not all_audio:
        display(HTML("<div style='background-color:#fff3cd;padding:15px;border-radius:8px;border-left:5px solid #ffc107;'><b style='color:#856404;'>⚠️ No audio tracks found.</b><br><span style='color:#856404;'>Generate some tracks first using the Web Interface (Block 4).</span></div>"))
    else:
        print(f"📁 Found {len(all_audio)} track(s):")
        print("-" * 70)
        
        total_size = 0
        for i, track in enumerate(sorted(all_audio, key=os.path.getctime, reverse=True), 1):
            size_mb = round(os.path.getsize(track) / (1024*1024), 2)
            total_size += size_mb
            filename = os.path.basename(track)
            created_time = time.strftime('%Y-%m-%d %H:%M:%S', time.localtime(os.path.getctime(track)))
            print(f"  {i}. {filename:<50} {size_mb:>6.2f} MB  [{created_time}]")
        
        print("-" * 70)
        print(f"📊 Total: {len(all_audio)} tracks, {round(total_size, 2)} MB")
        print()
        
        # Create zip archive
        zip_name = "/kaggle/working/YuE2_Generated_Tracks"
        print("📦 Creating zip archive...")
        shutil.make_archive(zip_name, 'zip', output_dir)
        
        zip_path = f"{zip_name}.zip"
        zip_size_mb = round(os.path.getsize(zip_path) / (1024*1024), 2)
        
        display(HTML(f"""
        <div style='background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; padding: 25px; border-radius: 15px; text-align: center; box-shadow: 0 10px 30px rgba(0,0,0,0.3); margin: 20px 0;'>
            <h2 style='margin: 0 0 15px 0; font-size: 1.8em;'>📦 Download All Tracks</h2>
            <p style='font-size: 1.1em; margin-bottom: 20px;'>Archive contains <b>{len(all_audio)}</b> tracks ({zip_size_mb} MB)</p>
            <a href='{zip_path}' download style='display: inline-block; background: white; color: #667eea; padding: 15px 35px; border-radius: 10px; text-decoration: none; font-weight: bold; font-size: 1.1em; box-shadow: 0 5px 15px rgba(0,0,0,0.2);'>
                ⬇️ Download ZIP Archive
            </a>
        </div>
        """))
        
        display(FileLink(zip_path, result_html_prefix="<div style='text-align:center; margin: 10px 0;'>"))